In [13]:
import os
import pprint
import getpass
import bs4
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
from dotenv import find_dotenv, load_dotenv
load_dotenv()

True

In [14]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import gradio as gr

In [15]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [16]:
print("Key loaded:", bool(os.getenv("OPENAI_API_KEY")))

Key loaded: True


In [17]:
def get_sitemap(url):
    req = Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0"}
    )
    response = urlopen(req)
    xml = BeautifulSoup(
        response,
        "lxml-xml",
        from_encoding=response.info().get_param("charset")
    )
    return xml

def get_urls(xml, name=None, data=None, verbose=False):
    urls = []
    for url in xml.find_all("url"):
        if xml.find("loc"):
            loc = url.findNext("loc").text
            urls.append(loc)
    return urls

url = "https://zerodha.com/varsity/chapter-sitemap2.xml"
xml = get_sitemap(url)
urls = get_urls(xml, verbose=False)

docs = []
# Loading just the first 10 for speed; adjust as needed
for i, url in enumerate(urls[:10]):
    loader = WebBaseLoader(url)
    docs.extend(loader.load())
    if i % 10 == 0:
        print("Loaded document index:", i)

/var/folders/b0/q4phnsjx7zn982yc82pq8ry40000gn/T/ipykernel_5659/1531516909.py:18: DeprecationWarning: Call to deprecated method findNext. (Replaced by find_next) -- Deprecated since version 4.0.0.
  loc = url.findNext("loc").text


Loaded document index: 0


In [18]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [19]:
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

In [20]:
print(f"Len docs: {len(docs)}, Len splits: {len(splits)}")

Len docs: 10, Len splits: 285


In [21]:
relevant_docs = retriever.invoke("What is a stock market index?")

In [22]:
relevant_docs

[Document(id='badda290-31d5-45e0-9c2a-6c7334f211a9', metadata={'language': 'en-US', 'source': 'https://zerodha.com/varsity/chapter/discounted-cash-flow-analysis-dcf/', 'title': 'DCF Valuation Model: Calculate Fair Share Price Step-by-Step', 'description': 'Build DCF model - calculate FCFF, apply WACC discount rate, compute terminal value, derive enterprise value, determine fair share price for investment.'}, page_content='The share price is Rs.300. What does this mean?\nThe price you see here is an outcome of the entire valuation exercise. We have made many assumptions here, and if these assumptions are made intelligently, then with some confidence, we can conclude that Rs.300 is the fair value of the stock. You can now compare the stock’s market value on the stock exchanges and decide to buy or wait. For example, if the stock is trading at Rs.425, then you know that it is overvalued compared to its fair value; hence you can avoid buying the stock.\nIf the stock is trading at Rs.225, t

In [23]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 

system_prompt = (
    "You are a financial assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "If the question is not clear ask follow up questions"
    "\n\n"
    "{context}"
)

In [ ]:
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)